Vision Transformer fine-tuné

In [ ]:
!pip install transformers pillow -q

In [ ]:
from google.colab import files
uploaded = files.upload()

!mkdir -p test_images
!mv *.png *.jpg *.jpeg *.webp *.bmp test_images/ 2>/dev/null
!ls test_images/

Saving Kamallah_Cambon_X_2024.png to Kamallah_Cambon_X_2024.png
'ADDI Yasmine 1 Snap_Insta_Telegram_ FBK.png'
'Cathedrale ND_Renault_X_2019.png'
'Covid 5G_Narce_FBK_2020.png'
 Hitler_Yael_X_2022.png
 Kamallah_Cambon_X_2024.png
'Le pape_Areg_Tiktok_IA_2023.png.jpg'
'Moranges_ Simpson Queen II_ Tiktok 2022.png'


In [ ]:
import torch
from transformers import pipeline
from PIL import Image
import os

# ============================================================
# MODÈLE : umm-maybe/AI-image-detector
#
# C'est un ViT (Vision Transformer) fine-tuné spécifiquement
# sur des milliers d'images réelles vs générées par IA
# (Stable Diffusion, Midjourney, DALL-E, etc.)
#
# Il a appris à reconnaître les artefacts invisibles à l'oeil
# que les générateurs IA laissent dans les pixels.
#
# Pas de texte, pas de règles manuelles, pas de prompts.
# Juste : image → réseau de neurones → score fake/réel.
# ============================================================

print("Chargement du modèle (première fois = téléchargement ~350MB)...")
detector = pipeline(
    "image-classification",
    model="umm-maybe/AI-image-detector",
    device=0 if torch.cuda.is_available() else -1
)
print("Modèle prêt !\n")

# Analyse
image_dir = "test_images"
results = []

print("=" * 75)
print(f"  {'FICHIER':<45} {'VERDICT':<12} {'CONFIANCE':>10}")
print("=" * 75)

for fname in sorted(os.listdir(image_dir)):
    if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')):
        path = os.path.join(image_dir, fname)
        img = Image.open(path).convert("RGB")

        preds = detector(img)

        # Extraire les scores
        scores = {p['label']: p['score'] for p in preds}
        prob_ai = scores.get('artificial', scores.get('ai', 0))
        prob_human = scores.get('human', scores.get('real', 0))

        # Si les labels sont différents, prendre le max
        if not prob_ai and not prob_human:
            top = preds[0]
            if 'artificial' in top['label'].lower() or 'ai' in top['label'].lower() or 'fake' in top['label'].lower():
                prob_ai = top['score']
                prob_human = 1 - top['score']
            else:
                prob_human = top['score']
                prob_ai = 1 - top['score']

        label = "🤖 IA" if prob_ai > prob_human else "📷 RÉEL"
        confiance = max(prob_ai, prob_human)
        results.append((fname, label, prob_ai, confiance))

        name = (fname[:42] + "...") if len(fname) > 45 else fname
        print(f"  {name:<45} {label:<12} {confiance:>9.1%}")

print("=" * 75)
nb_fake = sum(1 for r in results if "IA" in r[1])
nb_real = len(results) - nb_fake
print(f"\n  Résumé : {nb_fake} IA / {nb_real} réelle(s) sur {len(results)} image(s)")
print("=" * 75)

Chargement du modèle (première fois = téléchargement ~350MB)...


Loading weights:   0%|          | 0/449 [00:00<?, ?it/s]

Modèle prêt !

  FICHIER                                       VERDICT       CONFIANCE
  ADDI Yasmine 1 Snap_Insta_Telegram_ FBK.png   🤖 IA             94.1%
  Cathedrale ND_Renault_X_2019.png              🤖 IA             88.8%
  Covid 5G_Narce_FBK_2020.png                   📷 RÉEL           97.5%
  Hitler_Yael_X_2022.png                        🤖 IA             99.9%
  Kamallah_Cambon_X_2024.png                    🤖 IA             91.2%
  Le pape_Areg_Tiktok_IA_2023.png.jpg           🤖 IA             86.8%
  Moranges_ Simpson Queen II_ Tiktok 2022.png   📷 RÉEL           98.3%

  Résumé : 5 IA / 2 réelle(s) sur 7 image(s)
